# 🥈 Silver Layer - Fact Tables
## AWS S3 External Storage via Unity Catalog

**Purpose**: Build fact tables from Bronze layer with foreign keys to Silver dimensions

**Source**: `workspace.`workspace-adventureworks-bronze`` (S3)
**Target**: `workspace.`workspace-adventureworks-silver`` (S3)

**Fact Tables to Create**:
1. **fact_sales_order** - Sales order headers with customer/location FK
2. **fact_sales_order_detail** - Sales order line items with product FK
3. **fact_work_order** - Manufacturing work orders
4. **fact_purchase_order** - Purchase order headers and details

**Features**:
- Data quality validation
- Foreign key lookups to dimension surrogate keys
- Measure calculations (totals, extended amounts)
- Grain validation (ensuring proper granularity)

In [0]:
from pyspark.sql.functions import *
from pyspark.sql import Row
from datetime import datetime
import time

# Configuration
BRONZE_SCHEMA = "workspace.`workspace-adventureworks-bronze`"
SILVER_SCHEMA = "workspace.`workspace-adventureworks-silver`"

print("=" * 80)
print("🥈 SILVER LAYER - FACT TABLES")
print("=" * 80)
print(f"Start Time: {datetime.now()}")
print(f"Source: {BRONZE_SCHEMA} (AWS S3)")
print(f"Target: {SILVER_SCHEMA} (AWS S3)")
print("=" * 80)
print()

transformation_results = []

In [0]:
# Build Sales Order Header Fact
print("\n" + "=" * 80)
print("💰 BUILDING fact_sales_order")
print("=" * 80)

start_time = time.time()

try:
    # Read from Bronze
    sales_order_header = spark.table(f"{BRONZE_SCHEMA}.salesorderheader")
    
    # Read dimensions for FK lookups
    dim_customer = spark.table(f"{SILVER_SCHEMA}.dim_customer").filter(col("is_current") == True)
    dim_location = spark.table(f"{SILVER_SCHEMA}.dim_location").filter(col("is_current") == True)
    dim_date = spark.table(f"{SILVER_SCHEMA}.dim_date")
    
    # Build fact table with FK lookups
    fact_sales_order = (sales_order_header
        .join(
            dim_customer.select(col("dim_customer_sk"), col("customer_id")),
            sales_order_header.CustomerID == dim_customer.customer_id,
            "left"
        )
        .join(
            dim_location.select(col("dim_location_sk").alias("ship_to_location_sk"), col("address_id").alias("ship_address_id")),
            sales_order_header.ShipToAddressID == col("ship_address_id"),
            "left"
        )
        .join(
            dim_location.select(col("dim_location_sk").alias("bill_to_location_sk"), col("address_id").alias("bill_address_id")),
            sales_order_header.BillToAddressID == col("bill_address_id"),
            "left"
        )
        .join(
            dim_date.select(col("date_sk").alias("order_date_sk"), col("date_value").alias("order_date_value")),
            to_date(sales_order_header.OrderDate) == col("order_date_value"),
            "left"
        )
        .join(
            dim_date.select(col("date_sk").alias("due_date_sk"), col("date_value").alias("due_date_value")),
            to_date(sales_order_header.DueDate) == col("due_date_value"),
            "left"
        )
        .join(
            dim_date.select(col("date_sk").alias("ship_date_sk"), col("date_value").alias("ship_date_value")),
            to_date(sales_order_header.ShipDate) == col("ship_date_value"),
            "left"
        )
        .select(
            col("SalesOrderID").alias("sales_order_id"),
            col("SalesOrderNumber").alias("sales_order_number"),
            col("dim_customer_sk").alias("customer_sk"),
            col("order_date_sk"),
            col("due_date_sk"),
            col("ship_date_sk"),
            col("ship_to_location_sk"),
            col("bill_to_location_sk"),
            col("Status").alias("order_status"),
            col("OnlineOrderFlag").alias("is_online_order"),
            col("PurchaseOrderNumber").alias("purchase_order_number"),
            col("AccountNumber").alias("account_number"),
            col("ShipMethod").alias("ship_method"),
            col("SubTotal").alias("subtotal_amount"),
            col("TaxAmt").alias("tax_amount"),
            col("Freight").alias("freight_amount"),
            col("TotalDue").alias("total_due_amount"),
            col("Comment").alias("comment"),
            col("ModifiedDate").alias("modified_date"),
            col("sales_order_header.ingestion_timestamp"),
            col("sales_order_header.source_system")
        )
    )
    
    # Remove records where customer FK is null (orphaned records)
    initial_count = fact_sales_order.count()
    fact_sales_order = fact_sales_order.filter(col("customer_sk").isNotNull())
    final_count = fact_sales_order.count()
    
    print(f"   Data Quality Check:")
    print(f"     Initial: {initial_count:,} rows")
    print(f"     After FK validation: {final_count:,} rows")
    print(f"     Orphaned records removed: {initial_count - final_count:,}")
    
    # Write to Silver
    (fact_sales_order
        .write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(f"{SILVER_SCHEMA}.fact_sales_order")
    )
    
    duration = time.time() - start_time
    
    transformation_results.append({
        "table": "fact_sales_order",
        "status": "success",
        "row_count": final_count,
        "duration_seconds": round(duration, 2)
    })
    
    print(f"\n✅ fact_sales_order created: {final_count:,} rows in {duration:.2f}s")
    
except Exception as e:
    print(f"\n❌ Failed to create fact_sales_order: {str(e)}")
    transformation_results.append({"table": "fact_sales_order", "status": "failed", "row_count": 0, "duration_seconds": round(time.time() - start_time, 2)})

In [0]:
# Build Sales Order Detail Fact (Line Items)
print("\n" + "=" * 80)
print("📋 BUILDING fact_sales_order_detail")
print("=" * 80)

start_time = time.time()

try:
    # Read from Bronze
    sales_order_detail = spark.table(f"{BRONZE_SCHEMA}.salesorderdetail")
    
    # Read dimensions
    dim_product = spark.table(f"{SILVER_SCHEMA}.dim_product").filter(col("is_current") == True)
    fact_sales_order = spark.table(f"{SILVER_SCHEMA}.fact_sales_order")
    
    # Build fact table
    fact_sales_order_detail = (sales_order_detail
        .join(
            dim_product.select(col("dim_product_sk"), col("product_id")),
            sales_order_detail.ProductID == dim_product.product_id,
            "left"
        )
        .join(
            fact_sales_order.select(
                col("sales_order_id"),
                col("customer_sk"),
                col("order_date_sk")
            ),
            "SalesOrderID",
            "left"
        )
        .select(
            col("SalesOrderID").alias("sales_order_id"),
            col("SalesOrderDetailID").alias("sales_order_detail_id"),
            col("dim_product_sk").alias("product_sk"),
            col("customer_sk"),
            col("order_date_sk"),
            col("OrderQty").alias("order_quantity"),
            col("UnitPrice").alias("unit_price"),
            col("UnitPriceDiscount").alias("unit_price_discount"),
            (col("OrderQty") * col("UnitPrice") * (1 - col("UnitPriceDiscount"))).alias("line_total_amount"),
            col("ModifiedDate").alias("modified_date"),
            col("sales_order_detail.ingestion_timestamp"),
            col("sales_order_detail.source_system")
        )
    )
    
    # Data quality check
    initial_count = fact_sales_order_detail.count()
    fact_sales_order_detail = fact_sales_order_detail.filter(
        col("product_sk").isNotNull() & 
        col("customer_sk").isNotNull() &
        col("order_quantity") > 0
    )
    final_count = fact_sales_order_detail.count()
    
    print(f"   Data Quality Check:")
    print(f"     Initial: {initial_count:,} rows")
    print(f"     After validation: {final_count:,} rows")
    print(f"     Invalid records removed: {initial_count - final_count:,}")
    
    # Write to Silver
    (fact_sales_order_detail
        .write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(f"{SILVER_SCHEMA}.fact_sales_order_detail")
    )
    
    duration = time.time() - start_time
    
    transformation_results.append({
        "table": "fact_sales_order_detail",
        "status": "success",
        "row_count": final_count,
        "duration_seconds": round(duration, 2)
    })
    
    print(f"\n✅ fact_sales_order_detail created: {final_count:,} rows in {duration:.2f}s")
    
except Exception as e:
    print(f"\n❌ Failed to create fact_sales_order_detail: {str(e)}")
    transformation_results.append({"table": "fact_sales_order_detail", "status": "failed", "row_count": 0, "duration_seconds": round(time.time() - start_time, 2)})

In [0]:
# Build Work Order Fact
print("\n" + "=" * 80)
print("🏭 BUILDING fact_work_order")
print("=" * 80)

start_time = time.time()

try:
    # Read from Bronze
    work_order = spark.table(f"{BRONZE_SCHEMA}.workorder")
    
    # Read dimensions
    dim_product = spark.table(f"{SILVER_SCHEMA}.dim_product").filter(col("is_current") == True)
    dim_date = spark.table(f"{SILVER_SCHEMA}.dim_date")
    
    # Build fact table
    fact_work_order = (work_order
        .join(
            dim_product.select(col("dim_product_sk"), col("product_id")),
            work_order.ProductID == dim_product.product_id,
            "left"
        )
        .join(
            dim_date.select(col("date_sk").alias("start_date_sk"), col("date_value").alias("start_date_value")),
            to_date(work_order.StartDate) == col("start_date_value"),
            "left"
        )
        .join(
            dim_date.select(col("date_sk").alias("end_date_sk"), col("date_value").alias("end_date_value")),
            to_date(work_order.EndDate) == col("end_date_value"),
            "left"
        )
        .join(
            dim_date.select(col("date_sk").alias("due_date_sk"), col("date_value").alias("due_date_value")),
            to_date(work_order.DueDate) == col("due_date_value"),
            "left"
        )
        .select(
            col("WorkOrderID").alias("work_order_id"),
            col("dim_product_sk").alias("product_sk"),
            col("start_date_sk"),
            col("end_date_sk"),
            col("due_date_sk"),
            col("OrderQty").alias("order_quantity"),
            col("StockedQty").alias("stocked_quantity"),
            col("ScrappedQty").alias("scrapped_quantity"),
            col("ScrapReasonID").alias("scrap_reason_id"),
            col("ModifiedDate").alias("modified_date"),
            col("work_order.ingestion_timestamp"),
            col("work_order.source_system")
        )
    )
    
    # Data quality check
    initial_count = fact_work_order.count()
    fact_work_order = fact_work_order.filter(col("product_sk").isNotNull())
    final_count = fact_work_order.count()
    
    print(f"   Data Quality Check:")
    print(f"     Initial: {initial_count:,} rows")
    print(f"     After validation: {final_count:,} rows")
    
    # Write to Silver
    (fact_work_order
        .write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(f"{SILVER_SCHEMA}.fact_work_order")
    )
    
    duration = time.time() - start_time
    
    transformation_results.append({
        "table": "fact_work_order",
        "status": "success",
        "row_count": final_count,
        "duration_seconds": round(duration, 2)
    })
    
    print(f"\n✅ fact_work_order created: {final_count:,} rows in {duration:.2f}s")
    
except Exception as e:
    print(f"\n❌ Failed to create fact_work_order: {str(e)}")
    transformation_results.append({"table": "fact_work_order", "status": "failed", "row_count": 0, "duration_seconds": round(time.time() - start_time, 2)})

In [0]:
# Summary
print("\n" + "=" * 80)
print("📊 FACT TABLE TRANSFORMATION SUMMARY")
print("=" * 80)

success_count = sum(1 for r in transformation_results if r["status"] == "success")
failed_count = sum(1 for r in transformation_results if r["status"] == "failed")
total_rows = sum(r["row_count"] for r in transformation_results)

print(f"\nFact Tables Processed: {len(transformation_results)}")
print(f"  ✅ Success: {success_count}")
print(f"  ❌ Failed: {failed_count}")
print(f"Total Rows Created: {total_rows:,}")

summary_df = spark.createDataFrame([Row(**r) for r in transformation_results])
print("\nDetailed Results:")
display(summary_df.orderBy("status", "table"))

print("\n" + "=" * 80)
if failed_count == 0:
    print("✅ SILVER FACTS COMPLETED SUCCESSFULLY")
    print("=" * 80)
    print(f"Completion Time: {datetime.now()}")
    dbutils.notebook.exit("SUCCESS")
else:
    print("⚠️ SILVER FACTS COMPLETED WITH ERRORS")
    print("=" * 80)
    print(f"Completion Time: {datetime.now()}")
    dbutils.notebook.exit(f"PARTIAL_SUCCESS: {failed_count} facts failed")